# DX 704 Week 9 Project

This week's project will build an email spam classifier based on the Enron email data set.
You will perform your own feature extraction, and use naive Bayes to estimate the probability that a particular email is spam or not.
Finally, you will review the tradeoffs from different thresholds for automatically sending emails to the junk folder.

The full project description and a template notebook are available on GitHub: [Project 9 Materials](https://github.com/bu-cds-dx704/dx704-project-09).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download Data Set

We will be using the Enron spam data set as prepared in this GitHub repository.

https://github.com/MWiechmann/enron_spam_data

You may need to download this differently depending on your environment.

In [1]:
!wget https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip

zsh:1: command not found: wget


In [2]:
import os
# Ensure the working directory is the project folder so all file paths resolve correctly
project_dir = os.path.dirname(os.path.abspath("project.ipynb"))
os.chdir(project_dir)
print("Working directory:", os.getcwd())

Working directory: /Users/tetianakravchuk/Desktop/BUProjects/DX704-AI-in-the-Field/project-09-main


In [3]:
import pandas as pd

In [4]:
# pandas can read the zip file directly
enron_spam_data = pd.read_csv("enron_spam_data.zip")
enron_spam_data

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
...,...,...,...,...,...
33711,33711,= ? iso - 8859 - 1 ? q ? good _ news _ c = eda...,"hello , welcome to gigapharm onlinne shop .\np...",spam,2005-07-29
33712,33712,all prescript medicines are on special . to be...,i got it earlier than expected and it was wrap...,spam,2005-07-29
33713,33713,the next generation online pharmacy .,are you ready to rock on ? let the man in you ...,spam,2005-07-30
33714,33714,bloow in 5 - 10 times the time,learn how to last 5 - 10 times longer in\nbed ...,spam,2005-07-30


In [5]:
(enron_spam_data["Spam/Ham"] == "spam").mean()

0.5092834262664611

## Part 2: Design a Feature Extractor

Design a feature extractor for this data set and write out two files of features based on the text.
Don't forget that both the Subject and Message columns are relevant sources of text data.
For each email, you should count the number of repetitions of each feature present.
The auto-grader will assume that you are using a multinomial distribution in the following problems.

In [6]:
import re
import json
import numpy as np
from collections import Counter

def extract_features(subject, message):
    """Extract word count features from email subject and message."""
    text = ""
    if isinstance(subject, str):
        text += subject + " "
    if isinstance(message, str):
        text += message
    # Lowercase and keep only alphabetic tokens (length >= 2)
    words = re.findall(r'[a-z]{2,}', text.lower())
    return dict(Counter(words))

# Apply feature extraction to all emails
enron_spam_data['features'] = enron_spam_data.apply(
    lambda row: extract_features(row['Subject'], row['Message']), axis=1
)
enron_spam_data[['Message ID', 'Spam/Ham', 'features']].head()

,Message ID,Spam/Ham,features
0,0,ham,"{'christmas': 1, 'tree': 1, 'farm': 1, 'pictur..."
1,1,ham,"{'vastar': 6, 'resources': 4, 'inc': 4, 'gary'..."
2,2,ham,"{'calpine': 2, 'daily': 2, 'gas': 2, 'nominati..."
3,3,ham,"{'re': 2, 'issue': 4, 'fyi': 1, 'see': 1, 'not..."
4,4,ham,"{'meter': 3, 'nov': 3, 'allocation': 3, 'fyi':..."


Assign a row to the test data set if `Message ID % 30 == 0` and assign it to the training data set otherwise.
Write two files, "train-features.tsv" and "test-features.tsv" with two columns, Message ID and features_json.
The features_json column should contain a JSON dictionary where the keys are your feature names and the values are integer feature values.
This will give us a sparse feature representation.


In [7]:
# Split: test if Message ID % 30 == 0, else train
test_mask = enron_spam_data['Message ID'] % 30 == 0
train_data = enron_spam_data[~test_mask].copy()
test_data  = enron_spam_data[test_mask].copy()

print(f"Train: {len(train_data)}, Test: {len(test_data)}")

# Save train-features.tsv
train_features_df = train_data[['Message ID']].copy()
train_features_df['features_json'] = train_data['features'].apply(json.dumps)
train_features_df.to_csv('train-features.tsv', sep='\t', index=False)

# Save test-features.tsv
test_features_df = test_data[['Message ID']].copy()
test_features_df['features_json'] = test_data['features'].apply(json.dumps)
test_features_df.to_csv('test-features.tsv', sep='\t', index=False)

print("Saved train-features.tsv and test-features.tsv")

Train: 32592, Test: 1124
Saved train-features.tsv and test-features.tsv


Submit "train-features.tsv" and "test-features.tsv" in Gradescope.

Hint: these features will be graded based on the test accuracy of a logistic regression based on the training features.
This is to make sure that your feature set is not degenerate; you do not need to compute this regression yourself.
You can separately assess your feature quality based on your results in part 6.

## Part 3: Compute Conditional Probabilities

Based on your training data, compute appropriate conditional probabilities for use with naïve Bayes.
Use of additive smoothing with $\alpha=1$ to avoid zeros.


In [8]:
# Separate ham and spam training emails
train_ham  = train_data[train_data['Spam/Ham'] == 'ham']
train_spam = train_data[train_data['Spam/Ham'] == 'spam']

# Aggregate word counts across all ham / spam emails
ham_word_counts  = Counter()
spam_word_counts = Counter()
for feat in train_ham['features']:
    ham_word_counts.update(feat)
for feat in train_spam['features']:
    spam_word_counts.update(feat)

# Vocabulary: union of all words seen in training
vocab = set(ham_word_counts.keys()) | set(spam_word_counts.keys())
vocab_size = len(vocab)

total_ham_words  = sum(ham_word_counts.values())
total_spam_words = sum(spam_word_counts.values())

alpha = 1  # Laplace / additive smoothing

# Compute P(word | ham) and P(word | spam) for every vocabulary word
feature_probs_list = []
for word in vocab:
    ham_prob  = (ham_word_counts[word]  + alpha) / (total_ham_words  + alpha * vocab_size)
    spam_prob = (spam_word_counts[word] + alpha) / (total_spam_words + alpha * vocab_size)
    feature_probs_list.append({
        'feature': word,
        'ham_probability': ham_prob,
        'spam_probability': spam_prob,
    })

feature_probs_df = pd.DataFrame(feature_probs_list)
print(f"Vocabulary size: {vocab_size}")
feature_probs_df.head()

Vocabulary size: 141208


,feature,ham_probability,spam_probability
0,hqmslwte,2.463217e-07,6.146227e-07
1,cleveland,3.941147e-06,6.453538e-06
2,bombahakcx,2.463217e-07,6.146227e-07
3,invitamos,2.463217e-07,1.843868e-06
4,jl,3.694825e-06,3.380425e-06


Save the conditional probabilities in a file "feature-probabilities.tsv" with columns feature, ham_probability and spam_probability.

In [9]:
feature_probs_df.to_csv('feature-probabilities.tsv', sep='\t', index=False)
print("Saved feature-probabilities.tsv")

Saved feature-probabilities.tsv


Submit "feature-probabilities.tsv" in Gradescope.

## Part 4: Implement a Naïve Bayes Classifier

Implement a naïve Bayes classifier based on your previous feature probabilities.

In [10]:
# Prior log-probabilities from training class frequencies
n_train    = len(train_data)
log_prior_ham  = np.log(len(train_ham)  / n_train)
log_prior_spam = np.log(len(train_spam) / n_train)

# Fast lookups: word -> log P(word | class)
ham_log_prob  = dict(zip(feature_probs_df['feature'],
                         np.log(feature_probs_df['ham_probability'])))
spam_log_prob = dict(zip(feature_probs_df['feature'],
                         np.log(feature_probs_df['spam_probability'])))

def predict_proba(features):
    """Return (ham_prob, spam_prob) for a feature-count dict."""
    log_ham  = log_prior_ham
    log_spam = log_prior_spam
    for word, count in features.items():
        if word in ham_log_prob:          # skip out-of-vocabulary words
            log_ham  += count * ham_log_prob[word]
            log_spam += count * spam_log_prob[word]
    # Normalise with the log-sum-exp trick to avoid numerical issues
    log_max  = max(log_ham, log_spam)
    ham_exp  = np.exp(log_ham  - log_max)
    spam_exp = np.exp(log_spam - log_max)
    total    = ham_exp + spam_exp
    return ham_exp / total, spam_exp / total

# Apply to training data
train_preds = [
    {'Message ID': row['Message ID'],
     'ham':  predict_proba(row['features'])[0],
     'spam': predict_proba(row['features'])[1]}
    for _, row in train_data.iterrows()
]
train_preds_df = pd.DataFrame(train_preds)
train_preds_df.head()

,Message ID,ham,spam
0,1,1.0,3.224627e-166
1,2,1.0,1.415113e-12
2,3,1.0,4.771810e-149
3,4,1.0,9.275319e-143
4,5,1.0,6.106032e-38


Save your prediction probabilities to "train-predictions.tsv" with columns Message ID, ham and spam.

In [11]:
train_preds_df.to_csv('train-predictions.tsv', sep='\t', index=False)
print("Saved train-predictions.tsv")

Saved train-predictions.tsv


Submit "train-predictions.tsv" in Gradescope.

## Part 5: Predict Spam Probability for Test Data

Use your previous classifier to predict spam probability for the test data.

In [12]:
# Apply the same classifier to test data
test_preds = [
    {'Message ID': row['Message ID'],
     'ham':  predict_proba(row['features'])[0],
     'spam': predict_proba(row['features'])[1]}
    for _, row in test_data.iterrows()
]
test_preds_df = pd.DataFrame(test_preds)
test_preds_df.head()

,Message ID,ham,spam
0,0,0.058159,9.418406e-01
1,30,1.000000,2.111283e-79
2,60,1.000000,1.709230e-12
3,90,1.000000,7.266323e-33
4,120,1.000000,2.456632e-169


Save your prediction probabilities in "test-predictions.tsv" with the same columns as "train-predictions.tsv".

In [13]:
test_preds_df.to_csv('test-predictions.tsv', sep='\t', index=False)
print("Saved test-predictions.tsv")

Saved test-predictions.tsv


Submit "test-predictions.tsv" in Gradescope.

## Part 6: Construct ROC Curve

For every probability threshold from 0.01 to .99 in increments of 0.01, compute the false and true positive rates from the test data using the spam class for positives.
That is, if the predicted spam probability is greater than or equal to the threshold, predict spam.

In [14]:
# Merge test predictions with true labels
test_eval = test_data[['Message ID', 'Spam/Ham']].merge(test_preds_df, on='Message ID')
test_eval['true_spam'] = (test_eval['Spam/Ham'] == 'spam').astype(int)

total_pos = test_eval['true_spam'].sum()          # actual spam
total_neg = len(test_eval) - total_pos            # actual ham

roc_rows = []
for threshold in np.round(np.arange(0.01, 1.00, 0.01), 2):
    predicted_pos = test_eval['spam'] >= threshold
    tp = ( predicted_pos &  (test_eval['true_spam'] == 1)).sum()
    fp = ( predicted_pos &  (test_eval['true_spam'] == 0)).sum()
    tpr = tp / total_pos
    fpr = fp / total_neg
    roc_rows.append({'threshold': threshold,
                     'false_positive_rate': fpr,
                     'true_positive_rate':  tpr})

roc_df = pd.DataFrame(roc_rows)
roc_df.head(10)

,threshold,false_positive_rate,true_positive_rate
0,0.01,0.032609,0.996503
1,0.02,0.032609,0.996503
2,0.03,0.030797,0.996503
3,0.04,0.028986,0.996503
4,0.05,0.028986,0.994755
5,0.06,0.028986,0.994755
6,0.07,0.028986,0.994755
7,0.08,0.027174,0.994755
8,0.09,0.027174,0.994755
9,0.10,0.027174,0.994755


Save this data in a file "roc.tsv" with columns threshold, false_positive_rate and true_positive rate.

In [15]:
roc_df.to_csv('roc.tsv', sep='\t', index=False)
print("Saved roc.tsv")

Saved roc.tsv


Submit "roc.tsv" in Gradescope.

## Part 7: Signup for Gemini API Key

Create a free Gemini API key at https://aistudio.google.com/app/api-keys.
You will need to do this with a personal Google account - it will not work with your BU Google account.
This will not incur any charges unless you configure billing information for the key.

You will be asked to start a Gemini free trial for week 11.
This will not incur any charges unless you exceed expected usage by an order of magnitude.


No submission needed.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

Collaboration: none.

Additional libraries: none.

Generative AI usage: Claude Code (claude-sonnet-4-6 by Anthropic) was used to help implement this assignment. Claude assisted with designing the feature extractor, implementing the multinomial Naïve Bayes classifier with Laplace smoothing, and constructing the ROC curve. Claude Code does not generate shareable transcript links; see https://claude.ai/code for more information.